In [1]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
from dataclasses import dataclass
from pydantic import BaseModel
from typing import Literal

load_dotenv()

True

In [13]:
class StructuredOutput(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"]
    confidence: float
    reason: str

In [14]:
@dataclass(frozen=True)
class Provider:
    """Provider class to return the available provider based on the environment variable."""

    name: str
    env_var: str
    base_url: str
    model: str

In [15]:
PROVIDERS = [
    Provider(
        name="Gemini",
        env_var="GEMINI_API_KEY",
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        model="gemini-3.5-flash"
    ),
    Provider(
        name="Groq",
        env_var="GROQ_API_KEY",
        base_url="https://api.groq.com/openai/v1",
        model="openai/gpt-oss-20b"
    )
]

In [16]:
def select_provider() -> Provider:
    """Select the provider based on the environment variable."""
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider
    raise ValueError("No valid provider found. Please set the appropriate environment variable.")

In [17]:
def build_client(provider: Provider) -> OpenAI:
    """Build the OpenAI client based on the selected provider."""
    api_key = os.getenv(provider.env_var)
    if not api_key:
        raise ValueError(f"API key for {provider.name} is not set in environment variables.")
    
    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url
    )

In [18]:
SYSTEM_PROMPT = """
    You are a sentiment analysis model. Your task is to analyze the sentiment of the given iphone 
    reviews on a ecommerce site and provide a structured output in JSON format. 
    No extra text or comments.
    The output should contain the following fields:
    
    {
        "sentiment": "positive" | "negative" | "neutral",
        "confidence": float (0.0 to 1.0),
        "reason": "A brief explanation of why the sentiment was classified as such."
    }
""".strip()

In [19]:
def get_answer_from_gemini(prompt: str) -> str:
    """Get the answer from Gemini model."""
    provider = select_provider()
    client = build_client(provider)
    
    response = client.chat.completions.create(
        model=provider.model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    
    message = response.choices[0].message

    # Check if content is None (e.g., due to safety blocks)
    if not message or not message.content:
        finish_reason = response.choices[0].finish_reason
        raise ValueError(f"Model returned empty content. Finish reason: {finish_reason}")
        
    return message.content

In [20]:
REVIEWS = [
    "I love the new iPhone! The camera is amazing and the battery life is great.",
    "The iPhone is okay, but I expected more features for the price.",
    "I am very disappointed with the iPhone. It keeps freezing and the screen cracked easily."
]

In [21]:
positive_response = get_answer_from_gemini(REVIEWS[0])
print(positive_response)

{
    "sentiment": "positive",
    "confidence": 0.99,
    "reason": "The reviewer expresses strong positive emotions using words like 'love', 'amazing', and 'great' to describe the phone, its camera, and battery life."
}


In [22]:
def validate_sentiment_analysis(raw_reply: str) -> StructuredOutput:
    """Validate the sentiment analysis output against the StructuredOutput model."""
    try:
        parsed = json.loads(raw_reply)
        return StructuredOutput(**parsed)
    except (json.JSONDecodeError, ValueError) as e:
        raise ValueError(f"Invalid sentiment analysis output: {e}")

In [24]:
result = validate_sentiment_analysis(positive_response)
print(result)

sentiment='positive' confidence=0.99 reason="The reviewer expresses strong positive emotions using words like 'love', 'amazing', and 'great' to describe the phone, its camera, and battery life."
